# B2-022-probabilistic-latent-models — Practice p18 — Solution

**Type:** integrative · **Difficulty:** advanced · **Concepts:** autoencoder

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20260927
LOADER_CANDIDATES = (
    Path("../data/latent_data.py"),
    Path("units/B2-022-probabilistic-latent-models/data/latent_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("latent_data", loader_path)
latent_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(latent_data)
torch.set_num_threads(1)

## Solution

A linear autoencoder with MSE loss is optimized when the decoder's column space equals the top principal subspace of the centered training data (the decoder bias absorbs the mean).
The weights themselves are only determined up to an invertible $2\times2$ mixing, so the certificate compares orthogonal **projectors** $P_W=W(W^\top W)^{-1}W^\top$, which depend only on the column space.

In [ ]:
import copy

DATASET = "lowrank8d"


def train_rows():
    return latent_data.train_tensor(DATASET)


def heldout_rows():
    return latent_data.heldout_tensor(DATASET)


def split_hash_sets(name):
    hashes = latent_data.ROW_SHA256[name]
    train_part = {hashes[i] for i in latent_data.TRAIN_IDS[name]}
    heldout_part = {hashes[i] for i in latent_data.HELDOUT_IDS[name]}
    return train_part, heldout_part


TRAIN_HASHES, HELDOUT_HASHES = split_hash_sets(DATASET)


def install_training_row_guard(model):
    """Forward pre-hook: every row a training forward receives must hash into the training part."""
    seen = {"calls": 0, "rows": 0}

    def hook(module, args):
        batch = args[0]
        for row in batch:
            row_hash = latent_data.row_sha256(row)
            if row_hash not in TRAIN_HASHES or row_hash in HELDOUT_HASHES:
                raise AssertionError("a non-training row reached a training forward pass")
        seen["calls"] += 1
        seen["rows"] += batch.shape[0]

    return model.register_forward_pre_hook(hook), seen


class LinearAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Linear(8, 2)
        self.decoder = nn.Linear(2, 8)

    def forward(self, x):
        return self.decoder(self.encoder(x))


def train_linear_autoencoder(model, batch, optimizer):
    trace = []
    for _ in range(1500):
        optimizer.zero_grad(set_to_none=True)
        reconstruction = model(batch)
        loss = F.mse_loss(reconstruction, batch)
        trace.append(loss.item())
        loss.backward()
        optimizer.step()
    return trace


def projector(W):
    return W @ torch.linalg.solve(W.T @ W, W.T)


T = train_rows().double()
centered = T - T.mean(dim=0)
_, _, Vh = torch.linalg.svd(centered, full_matrices=False)
W_pca = Vh[:2].T
P_pca = projector(W_pca)


def subspace_distance(model):
    W_ae = model.decoder.weight.detach().double()
    return torch.linalg.norm(projector(W_ae) - P_pca).item()


torch.manual_seed(SEED)
model = LinearAutoencoder()
optimizer = torch.optim.Adam(model.parameters(), lr=0.02, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)

# (A)
H = heldout_rows()
distance_before = subspace_distance(model)
with torch.no_grad():
    heldout_mse_before = F.mse_loss(model(H), H).item()
decoder_weight_before = copy.deepcopy(model.decoder.weight.detach())

# (B)
guard, guard_stats = install_training_row_guard(model)
trace = train_linear_autoencoder(model, train_rows(), optimizer)
guard.remove()
distance_after = subspace_distance(model)
with torch.no_grad():
    heldout_mse_after = F.mse_loss(model(H), H).item()
W_ae = model.decoder.weight.detach().double()
P_ae = projector(W_ae)
print(f"distance before={distance_before:.4f} after={distance_after:.6f}")
print(f"held-out MSE before={heldout_mse_before:.4f} after={heldout_mse_after:.5f}; final train loss={trace[-1]:.5f}")

### (D) Why projectors, and why centering

Even a perfectly trained linear autoencoder only pins down the 2-D **column space** of the decoder: for any invertible $2\times2$ matrix $A$, replacing the decoder weight $W$ by $WA$ and the encoder by $A^{-1}$ times the encoder gives the identical reconstruction, so $W_{ae}$ is in general neither orthonormal nor aligned with (or even sign-matched to) the singular vectors in $W_{pca}$, and `torch.allclose(W_ae, W_pca)` would fail for a correct model; the projector $P_W$ is invariant to $W\mapsto WA$ and is the right object to compare.
The PCA reference must be centered because the autoencoder has biases: the decoder bias models the data mean, so the weights capture only the directions of variation about the mean, which are the top singular vectors of the *centered* matrix; the uncentered SVD would instead be pulled toward the mean vector itself and name a different subspace.

### Answer check

In [ ]:
assert train_rows().shape == (48, 8) and H.shape == (12, 8)
assert train_rows().dtype == torch.float32 and H.dtype == torch.float32
assert len(trace) == 1500
assert distance_before >= 0.5
assert distance_after <= 0.02
assert heldout_mse_after <= 0.03
assert torch.allclose(P_ae, P_ae.T, atol=1e-8, rtol=1e-8)
assert torch.allclose(P_ae @ P_ae, P_ae, atol=1e-8, rtol=1e-8)
assert not torch.equal(model.decoder.weight.detach(), decoder_weight_before)
assert guard_stats == {"calls": 1500, "rows": 1500 * 48}
assert all(latent_data.row_sha256(row) in TRAIN_HASHES for row in train_rows())
assert not any(latent_data.row_sha256(row) in HELDOUT_HASHES for row in train_rows())
assert abs(P_pca.trace().item() - 2.0) <= 1e-8